# TotalSegmentator inference (nb2): converted NIfTI → segmentations  —  model-specific

Runs TotalSegmentator (v1.5.6) on the GPU VM. Consumes the
**Boundary-A** archive `converted_nifti.tar.lz4` from nb1 and emits the **Boundary-B**
archive `segmentations.tar.lz4` with the canonical layout:
```
<SeriesInstanceUID>/<model>/segmentations/<SeriesInstanceUID>.nii.gz
<SeriesInstanceUID>/<model>/label_map.json      # {label_id: label_name}
```

Supported tasks (selected via the `task` papermill parameter):

- **`total`** (default): 104-structure full-body segmentation (`--ml` multilabel).
  Label IDs from `class_map['total']`.
- **`lung_vessels`**: two-step pipeline — a fast total pre-segmentation
  (for lung-lobe crop masks), then the dedicated Task258 lung-vessel model.
  `--ml` and `--fast` are incompatible with this task in v1.5.6.
  Label IDs from `class_map['lung_vessels']`.

nb3 (shared) turns the output into DICOM-SEG + radiomics + SR using the
SNOMED mapping rows that match each task’s label names.

## Imports

In [ ]:
import json
import shutil
import subprocess
import time
import traceback
from pathlib import Path

NOTEBOOK_START = time.time()
def _elapsed(s=None):
    return f"{time.time() - (s if s is not None else NOTEBOOK_START):.1f}s"
print(f"[T+{_elapsed()}] Imports complete")

## Parameters

In [ ]:
# Boundary-A archive produced by nb1 (local file on the same VM).
converted_nifti_path = "converted_nifti.tar.lz4"

# Short model identifier used in the Boundary-B layout (<uid>/<model>/...).
model_name = "total"

# 'cuda' for GPU, 'cpu' for CPU-only.
accelerator = "cuda"

# Reserved for checkpoint/resume on preemption (not yet wired in this notebook).
checkpoint_gcs = ""

# Model-specific knobs injected via `papermill -f inference_params.yaml`.
# fast=True uses TotalSegmentator's 3mm model (faster, lower resolution).
fast = False

# TotalSegmentator task: 'total' (default 104-structure) or 'lung_vessels'.
task = "total"

## Extract Boundary-A archive

In [ ]:
NIFTI_DIR = Path('/tmp/converted_nifti')
SEG_DIR = Path('/tmp/segmentations')
for _d in (NIFTI_DIR, SEG_DIR):
    if _d.exists():
        shutil.rmtree(_d)
    _d.mkdir(parents=True, exist_ok=True)

subprocess.run(f'lz4 -d -c {converted_nifti_path} | tar -xf - -C {NIFTI_DIR.parent}',
               shell=True, check=True)
if (NIFTI_DIR / 'converted_nifti').is_dir():
    NIFTI_DIR = NIFTI_DIR / 'converted_nifti'
series_uids = sorted(d.name for d in NIFTI_DIR.iterdir() if d.is_dir())
print(f'Series : {len(series_uids)}  |  task={task}  fast={fast}')

## Authoritative label map from TotalSegmentator's class map

In [ ]:
from totalsegmentator.map_to_binary import class_map

VALID_TASKS = ('total', 'lung_vessels')
if task not in VALID_TASKS:
    raise ValueError(f"Unknown task '{task}'; valid tasks: {VALID_TASKS}")

TASK_LABELS = {str(k): v for k, v in class_map[task].items()}
print(f'Loaded {len(TASK_LABELS)} TotalSegmentator label ids for task={task}')

## Run TotalSegmentator → Boundary-B layout

In [ ]:
import nibabel as nib
import numpy as np

errors = []
usage_metrics = {'series': {}}

def _run_total(nii, work, fast):
    """Run TotalSegmentator 'total' task (--ml multilabel). Returns the produced .nii path."""
    out_dir = work / 'segmentations'
    cmd = ['TotalSegmentator', '-i', str(nii), '-o', str(out_dir), '--ml']
    if fast:
        cmd.append('--fast')
    print(f'  {" ".join(cmd)}', flush=True)
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if res.returncode != 0:
        raise RuntimeError(f'TotalSegmentator rc={res.returncode}\n{res.stderr}')
    produced = next(iter(sorted(work.rglob('*.nii'))), None)
    if produced is None:
        raise RuntimeError('no multilabel NIfTI produced')
    return produced

def _run_lung_vessels(nii, work):
    """Two-step lung_vessels pipeline: fast total pre-seg, then lung_vessels inference, then multilabel combine."""
    pre_dir = work / 'pre_seg'
    pre_dir.mkdir(parents=True, exist_ok=True)

    # Step 1: fast total to get lung lobe masks for cropping (3mm is fine for spatial crop)
    pre_cmd = ['TotalSegmentator', '-i', str(nii), '-o', str(pre_dir), '--fast']
    print(f'  Step 1/2 (pre-seg): {" ".join(pre_cmd)}', flush=True)
    pre_res = subprocess.run(pre_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if pre_res.returncode != 0:
        raise RuntimeError(f'pre-segmentation rc={pre_res.returncode}\n{pre_res.stderr}')

    required = ['lung_upper_lobe_left', 'lung_lower_lobe_left', 'lung_upper_lobe_right',
                'lung_middle_lobe_right', 'lung_lower_lobe_right']
    missing = [m for m in required if not (pre_dir / f'{m}.nii.gz').exists()]
    if missing:
        raise RuntimeError(f'pre-segmentation missing lung masks: {missing}')

    # Step 2: lung_vessels inference (same output dir so it finds the lung lobe masks)
    lv_cmd = ['TotalSegmentator', '-i', str(nii), '-o', str(pre_dir), '--task', 'lung_vessels']
    print(f'  Step 2/2 (lung_vessels): {" ".join(lv_cmd)}', flush=True)
    lv_res = subprocess.run(lv_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if lv_res.returncode != 0:
        raise RuntimeError(f'lung_vessels rc={lv_res.returncode}\n{lv_res.stderr}')

    # Step 3: combine individual binary masks into a single multilabel volume
    ref_img = None
    multilabel = None
    for label_id, label_name in class_map[task].items():
        mask_path = pre_dir / f'{label_name}.nii.gz'
        if not mask_path.exists():
            print(f'  WARNING: expected mask {mask_path.name} not found, skipping label {label_id}')
            continue
        img = nib.load(str(mask_path))
        if ref_img is None:
            ref_img = img
            multilabel = np.zeros(img.shape, dtype=np.uint8)
        multilabel[np.asanyarray(img.dataobj) > 0.5] = label_id

    if ref_img is None:
        raise RuntimeError('no lung_vessels output masks found')
    return nib.Nifti1Image(multilabel, ref_img.affine, ref_img.header)

for uid in series_uids:
    nii = NIFTI_DIR / uid / f'{uid}.nii.gz'
    if not nii.exists():
        cands = list((NIFTI_DIR / uid).glob('*.nii.gz'))
        if not cands:
            errors.append(f'{uid}: no NIfTI found')
            continue
        nii = cands[0]
    work = Path('/tmp/ts_work') / uid
    if work.exists():
        shutil.rmtree(work)
    work.mkdir(parents=True, exist_ok=True)
    print(f'[T+{_elapsed()}] {uid} (task={task}):', flush=True)
    t0 = time.time()
    try:
        dest = SEG_DIR / uid / model_name / 'segmentations'
        dest.mkdir(parents=True, exist_ok=True)

        if task == 'total':
            produced = _run_total(nii, work, fast)
            subprocess.run(f'gzip -c "{produced}" > "{dest / (uid + ".nii.gz")}"',
                           shell=True, check=True)
        elif task == 'lung_vessels':
            if fast:
                raise ValueError("task 'lung_vessels' is incompatible with fast=True")
            ml_img = _run_lung_vessels(nii, work)
            nib.save(ml_img, str(dest / (uid + '.nii.gz')))

        (SEG_DIR / uid / model_name / 'label_map.json').write_text(
            json.dumps({'model': model_name, 'labels': TASK_LABELS}, indent=2))
        shutil.copy(str(nii), str(SEG_DIR / uid / 'reference.nii.gz'))
        usage_metrics['series'][uid] = {'model_inference_s': round(time.time() - t0, 1)}
        print(f'  done in {usage_metrics["series"][uid]["model_inference_s"]}s')
    except Exception as exc:
        errors.append(f'{uid}: {traceback.format_exc()}')
        print(f'  ERROR: {exc}')
    finally:
        shutil.rmtree(work, ignore_errors=True)

if errors:
    Path('inference_errors.txt').write_text('\n'.join(errors))
print(f'[T+{_elapsed()}] Inference complete ({len(errors)} error(s))')

## Package Boundary-B archive + usage metrics

In [ ]:
import csv
produced = [d for d in SEG_DIR.iterdir() if d.is_dir()]
if not produced:
    raise RuntimeError('No segmentations produced — see inference_errors.txt')

subprocess.run(f'tar -cf - -C {SEG_DIR.parent} {SEG_DIR.name} | lz4 > segmentations.tar.lz4',
               shell=True, check=True)
size_mb = Path('segmentations.tar.lz4').stat().st_size / (1024 ** 2)

usage_metrics['total_elapsed_s'] = round(time.time() - NOTEBOOK_START, 1)
with open('inference_UsageMetrics.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['SeriesInstanceUID', 'model', 'model_inference_s', 'run_total_elapsed_s'])
    for uid, m in usage_metrics['series'].items():
        w.writerow([uid, model_name, m.get('model_inference_s', ''), usage_metrics['total_elapsed_s']])

print(f'[T+{_elapsed()}] Wrote segmentations.tar.lz4 ({size_mb:.1f} MB, {len(produced)} series)')